In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys

base_path = "../../"
sys.path.append(base_path)

## Load dataset

Use `MetadataFilter` to explore and download samples without fetching the full dataset.
Place `metadata.parquet` in your dataset directory before running, for example at `datasets/pyvista/metadata.parquet`.

In [ ]:
from cooldata.metadata import MetadataFilter

f = MetadataFilter("datasets/pyvista/metadata.parquet")
f.summary()

## Filter and download samples

Filter by metadata parameters and download only the samples you need.
All filters are chainable. Call `.reset()` between separate filter chains.

In [ ]:
# Download 12 random samples
ds_pv = f.load_random(n=12, seed=42, data_dir="datasets/pyvista")

In [ ]:
# Or filter by metadata first — e.g. high velocity, at least 3 active bodies
# ds_pv = (
#     f.velocity(min=4.0)
#      .n_bodies(min=3)
#      .load(num_samples=12, data_dir="datasets/pyvista")
# )
# f.reset()

# Or download specific design IDs
# ds_pv = f.load_by_ids([125002, 125037, 125103], data_dir="datasets/pyvista")

# Or download by run
# ds_pv = f.load_by_run("run_1", num_samples=12, data_dir="datasets/pyvista")

## Inspect sample metadata

In [ ]:
# SVG preview of a sample — shows bounding box, quads, and cylinders
ds_pv[11]

In [ ]:
# Full metadata: velocity, quad and cylinder parameters
ds_pv[4].metadata

In [ ]:
# Check how many samples match a filter without downloading
print(f.velocity(min=4.0).n_bodies(min=3).count(), "samples match")
f.reset()

# Get matching metadata as a DataFrame
df = f.velocity(min=3.0, max=5.0).temperature(body=1, min=60.0).get_dataframe()
f.reset()
df.head()

## Visualize flow fields

Available volume fields: `Velocity_0`, `Velocity_1`, `Velocity_2`, `Pressure`, `Temperature`, `TurbulentKineticEnergy`, `TurbulentDissipationRate`

Available surface fields: `Pressure`, `Temperature`, `HeatTransferCoefficient`, `WallShearStressMagnitude`, `WallShearStress_0/1/2`, `Normal_0/1/2`, `AreaMagnitude`

In [ ]:
ds_pv[11].plot_volume("TurbulentDissipationRate")

In [ ]:
ds_pv[4].plot_surface("Temperature")

## Access raw surface data

In [ ]:
# surface_data[0] = first block group, [1] = second surface block
ds_pv[4].surface_data[0][1]

## Voxelized representation

Convert samples to a voxel grid for image-based ML approaches.

In [ ]:
from cooldata.voxel_flow_field_dataset import (
    VoxelFlowFieldDataset,
    VoxelFlowFieldDatasetConfig,
)

ds_voxel = VoxelFlowFieldDataset(
    "datasets/voxel",
    VoxelFlowFieldDatasetConfig(pyvista_dataset=ds_pv, resolution=(32, 16, 16)),
)

In [ ]:
ds_voxel[11].plot_slice_interactively("Velocity")